# LUMOS Tutorial

This notebook walks through two end-to-end examples of L0 regularization pruning with LUMOS:

1. **MNIST** — MLP (784→512→256→128→10), flat dataset, quick to run.
2. **CIFAR-10** — 2× width ResNet18, image classification, production recipe.

Both examples use `--target-prune-ratio` early-stop to prevent the accuracy collapse that
occurs when L0 gates are driven too negative before fine-tuning begins.

## Setup

In [1]:
import os, sys

# Make sure we run from the LUMOS directory
LUMOS_DIR = os.path.dirname(os.path.abspath("__file__"))
if os.path.basename(LUMOS_DIR) != "LUMOS":
    LUMOS_DIR = os.path.join(LUMOS_DIR, "LUMOS")
os.chdir(LUMOS_DIR)
if LUMOS_DIR not in sys.path:
    sys.path.insert(0, LUMOS_DIR)

print("Working directory:", os.getcwd())

Working directory: /nfs/hpc/share/gaosho/projects/loxia/LOXIA_dev/LUMOS


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from lumos.pruner import L0Pruner
from lumos.models import get_model
from lumos.data import get_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/nfs/stak/users/gaosho/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


---
## Example 1 — MNIST + MLP

Model: `MnistMLP` — input images are flattened from `(N,1,28,28)` to `(N,784)` inside `forward()`.

Architecture: **784 → 512 → 256 → 128 → 10**

Goal: prune_ratio ≥ 0.40, val_acc ≥ 95%

**Verified result (2026-05-23):**
- prune_ratio = **0.4033** (early-stop at epoch 11)
- best val acc = **98.25%** (after 30 fine-tune epochs)
- Input pixels pruned (layers.0): **394/784 = 50.3%**

### 1-A  Load data and model

In [3]:
MNIST_OUTPUT = "./output/tutorial_mnist"
os.makedirs(MNIST_OUTPUT, exist_ok=True)

# Dataset  (downloads to ./data/torchdata if not present)
num_classes, train_dst, val_dst, input_size = get_dataset(
    "mnist", data_root="./data", batch_size=256
)
trainloader = torch.utils.data.DataLoader(train_dst, batch_size=256, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(val_dst,   batch_size=256, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dst)}, Test samples: {len(val_dst)}, Classes: {num_classes}")

# Model
model = get_model("mnist_mlp", num_classes=num_classes).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

Train samples: 60000, Test samples: 10000, Classes: 10
Model parameters: 567,434
MnistMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=10, bias=True)
  )
)


### 1-B  Initialise L0 Pruner

In [4]:
N = float(len(train_dst))          # 60 000
WEIGHT_DECAY = 5e-4
LAMBA = 0.002                      # L0 sparsity penalty per gate
TARGET_PRUNE_RATIO = 0.40          # stop L0 training once 40% channels are pruned
FINETUNE_EPOCHS = 30

pruner = L0Pruner(
    model,
    droprate_init=0.5,
    temperature=2.0 / 3.0,
    weight_decay=N * WEIGHT_DECAY,  # scale by N as in original paper
    lamba=LAMBA,
    lamda=N,
    device=DEVICE,
    sparse_training=True,
    writer_dir=os.path.join(MNIST_OUTPUT, "tensorboard"),
)

print(f"Pruner ready. Gated layers: {len(pruner.masked_layers)}")

2026-05-27 12:45:21.827763: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-27 12:45:22.456051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779911122.672818 4164416 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779911122.750872 4164416 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-27 12:45:23.283680: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'L0Pruner' object has no attribute 'masked_layers'

### 1-C  Optimizer and scheduler

In [ ]:
LR      = 0.01
EPOCHS  = 50

gate_params   = [p for n, p in model.named_parameters() if "qz_loga" in n]
weight_params = [p for n, p in model.named_parameters() if "qz_loga" not in n]

optimizer = optim.Adam(
    [{"params": weight_params, "lr": LR},
     {"params": gate_params,   "lr": LR * 1.0}],  # gate_lr must equal weight lr (not 0.1x)
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

### 1-D  Training loop with early-stop

In [ ]:
def accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


history = {"epoch": [], "val_acc": [], "prune_ratio": []}
best_acc = 0.0

print("=== L0 Training ===")
for epoch in range(EPOCHS):
    model.train()
    for x, y in trainloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y) + pruner.regularize() / N
        loss.backward()
        optimizer.step()
        pruner.constrain_parameters()  # clamp qz_loga to prevent extreme values

    val_acc     = accuracy(model, testloader, DEVICE)
    prune_ratio = pruner.get_prune_ratio()
    scheduler.step()
    # Pin gate_lr constant — prevents CosineAnnealingLR from decaying it to zero
    optimizer.param_groups[1]["lr"] = LR

    history["epoch"].append(epoch)
    history["val_acc"].append(val_acc)
    history["prune_ratio"].append(prune_ratio)
    print(f"Epoch {epoch:3d} | Val Acc: {val_acc:.2f}% | Prune Ratio: {prune_ratio:.4f}")

    if TARGET_PRUNE_RATIO > 0 and prune_ratio >= TARGET_PRUNE_RATIO:
        print(f"\n>>> Target prune ratio {TARGET_PRUNE_RATIO} reached. Early-stopping L0 training.")
        break

print("\n=== Merging mask → Fine-tuning ===")
pruner.merge_mask()

# Record which input columns are live so pruned features cannot regrow
keep_masks = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        per_feat = module.weight.data.abs().max(dim=0)[0]
        keep_masks[name] = (per_feat >= 1e-6)

ft_params    = [p for n, p in model.named_parameters() if 'qz_loga' not in n]
ft_optimizer = optim.Adam(ft_params, lr=LR * 0.1)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FINETUNE_EPOCHS)

for ft_epoch in range(FINETUNE_EPOCHS):
    model.train()
    for x, y in trainloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ft_optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        ft_optimizer.step()
        # Re-zero pruned input columns to prevent regrowth
        with torch.no_grad():
            for name, module in model.named_modules():
                if name in keep_masks and isinstance(module, nn.Linear):
                    module.weight.data[:, ~keep_masks[name]] = 0.0

    val_acc = accuracy(model, testloader, DEVICE)
    ft_scheduler.step()

    history["epoch"].append(epoch + 1 + ft_epoch)
    history["val_acc"].append(val_acc)
    history["prune_ratio"].append(pruner.get_prune_ratio())
    print(f"Finetune Epoch {ft_epoch:3d} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(MNIST_OUTPUT, "best_model.pth"))

print(f"\nBest val acc: {best_acc:.2f}%  |  Final prune ratio: {pruner.get_prune_ratio():.4f}")


### 1-E  Results

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.plot(history["epoch"], history["val_acc"],    color="steelblue", label="Val Acc (%)")
ax2.plot(history["epoch"], history["prune_ratio"], color="darkorange", linestyle="--", label="Prune Ratio")

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Val Accuracy (%)",  color="steelblue")
ax2.set_ylabel("Prune Ratio",        color="darkorange")
ax2.set_ylim(0, 1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
plt.title("MNIST MLP — L0 Pruning + Fine-Tuning")
plt.tight_layout()
plt.savefig(os.path.join(MNIST_OUTPUT, "training_curve.png"), dpi=150)
plt.show()

---
## Example 2 — CIFAR-10 + 2× Width ResNet18

Model: `resnet18_2x` — ResNet18 with 2× channel width at every layer, yielding ~4× more parameters.

Goal: prune_ratio ≥ 0.50, val_acc ≥ 85%

**Verified result (run `run_2x_earlystop`, 2026-05-22):**
- prune_ratio = **0.5049**, best val acc = **95.28%**

### 2-A  Load data and model

In [ ]:
CIFAR_OUTPUT = "./output/tutorial_cifar10"
os.makedirs(CIFAR_OUTPUT, exist_ok=True)

num_classes_c, train_dst_c, val_dst_c, _ = get_dataset(
    "cifar10", data_root="./data", batch_size=128
)
trainloader_c = torch.utils.data.DataLoader(train_dst_c, batch_size=128, shuffle=True,  num_workers=2)
testloader_c  = torch.utils.data.DataLoader(val_dst_c,   batch_size=128, shuffle=False, num_workers=2)

print(f"Train: {len(train_dst_c)}, Test: {len(val_dst_c)}, Classes: {num_classes_c}")

model_c = get_model("resnet18_2x", num_classes=num_classes_c).to(DEVICE)
total_params_c = sum(p.numel() for p in model_c.parameters())
print(f"Model parameters: {total_params_c:,}")

### 2-B  Initialise L0 Pruner

In [ ]:
N_c            = float(len(train_dst_c))   # 50 000
LAMBA_C        = 0.002
TARGET_RATIO_C = 0.50
FINETUNE_C     = 100

pruner_c = L0Pruner(
    model_c,
    droprate_init=0.5,
    temperature=2.0 / 3.0,
    weight_decay=N_c * 5e-4,
    lamba=LAMBA_C,
    lamda=N_c,
    device=DEVICE,
    sparse_training=True,
    writer_dir=os.path.join(CIFAR_OUTPUT, "tensorboard"),
)
print(f"Pruner ready. Gated layers: {len(pruner_c.masked_layers)}")

### 2-C  Optimizer and scheduler

In [ ]:
LR_C    = 0.1
EPOCHS_C = 200   # upper bound; early-stop will trigger well before this

gate_params_c   = [p for n, p in model_c.named_parameters() if "qz_loga" in n]
weight_params_c = [p for n, p in model_c.named_parameters() if "qz_loga" not in n]
gate_lr_c = LR_C * 1.0   # gate lr = full lr (matches run_2x_earlystop config)

epoch_drop_c = [int(0.6 * EPOCHS_C), int(0.8 * EPOCHS_C), int(0.9 * EPOCHS_C)]

optimizer_c = optim.SGD(
    [{"params": weight_params_c, "lr": LR_C},
     {"params": gate_params_c,   "lr": gate_lr_c}],
    momentum=0.9, weight_decay=0, nesterov=True,
)
scheduler_c = optim.lr_scheduler.MultiStepLR(optimizer_c, milestones=epoch_drop_c, gamma=0.2)
criterion_c = nn.CrossEntropyLoss()

print(f"SGD lr={LR_C}, gate_lr={gate_lr_c}, milestones={epoch_drop_c}")

### 2-D  Training loop with early-stop

> **Why early-stop matters for CIFAR-10 / ResNet:**
>
> During L0 training, validation uses *hard* (binarized) gates.  As the L0 penalty pushes
> gate log-alphas negative, channels are effectively zeroed out and val accuracy collapses
> to near-random (10–20%) even while train accuracy stays at ~88%.  By stopping the moment
> the prune ratio hits 50%, the surviving channels still hold well-trained weights, so
> `merge_mask()` + fine-tuning recovers cleanly.

In [ ]:
def accuracy_c(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    return 100.0 * correct / total


history_c  = {"epoch": [], "val_acc": [], "prune_ratio": []}
best_acc_c = 0.0
l0_stop_epoch = None

print("=== L0 Training (CIFAR-10) ===")
for epoch in range(EPOCHS_C):
    model_c.train()
    for x, y in trainloader_c:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer_c.zero_grad()
        loss = criterion_c(model_c(x), y) + pruner_c.regularize() / N_c
        loss.backward()
        optimizer_c.step()
        pruner_c.constrain_parameters()  # clamp qz_loga to prevent extreme values

    val_acc_c     = accuracy_c(model_c, testloader_c, DEVICE)
    prune_ratio_c = pruner_c.get_prune_ratio()
    scheduler_c.step()
    # Pin gate_lr constant — prevents MultiStepLR from decaying it at milestones
    optimizer_c.param_groups[1]["lr"] = gate_lr_c

    history_c["epoch"].append(epoch)
    history_c["val_acc"].append(val_acc_c)
    history_c["prune_ratio"].append(prune_ratio_c)

    if epoch % 10 == 0 or prune_ratio_c >= TARGET_RATIO_C:
        print(f"Epoch {epoch:3d} | Val Acc: {val_acc_c:.2f}% | Prune Ratio: {prune_ratio_c:.4f}")

    if TARGET_RATIO_C > 0 and prune_ratio_c >= TARGET_RATIO_C:
        l0_stop_epoch = epoch
        print(f"\n>>> Target prune ratio {TARGET_RATIO_C} reached at epoch {epoch}. Early-stopping.")
        break

print("\n=== Merging mask → Fine-tuning ===")
pruner_c.merge_mask()

# Record which channels are live so pruned channels cannot regrow
keep_masks_c = {}
for name, module in model_c.named_modules():
    if isinstance(module, nn.Conv2d):
        per_ch = module.weight.data.view(module.weight.size(0), -1).abs().max(dim=1)[0]
        keep_masks_c[name] = (per_ch >= 1e-6)
    elif isinstance(module, nn.Linear):
        per_feat = module.weight.data.abs().max(dim=0)[0]
        keep_masks_c[name] = (per_feat >= 1e-6)

ft_params_c    = [p for n, p in model_c.named_parameters() if 'qz_loga' not in n]
ft_optimizer_c = optim.SGD(ft_params_c, lr=LR_C * 0.1, momentum=0.9, nesterov=True)
ft_scheduler_c = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer_c, T_max=FINETUNE_C)
l0_stop_epoch  = l0_stop_epoch or EPOCHS_C

for ft_epoch in range(FINETUNE_C):
    model_c.train()
    for x, y in trainloader_c:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ft_optimizer_c.zero_grad()
        criterion_c(model_c(x), y).backward()
        ft_optimizer_c.step()
        # Re-zero pruned channels to prevent regrowth
        with torch.no_grad():
            for name, module in model_c.named_modules():
                if name in keep_masks_c:
                    if isinstance(module, nn.Conv2d):
                        module.weight.data[~keep_masks_c[name]] = 0.0
                    elif isinstance(module, nn.Linear):
                        module.weight.data[:, ~keep_masks_c[name]] = 0.0

    val_acc_c = accuracy_c(model_c, testloader_c, DEVICE)
    ft_scheduler_c.step()

    history_c["epoch"].append(l0_stop_epoch + 1 + ft_epoch)
    history_c["val_acc"].append(val_acc_c)
    history_c["prune_ratio"].append(pruner_c.get_prune_ratio())

    if ft_epoch % 10 == 0:
        print(f"Finetune Epoch {ft_epoch:3d} | Val Acc: {val_acc_c:.2f}%")

    if val_acc_c > best_acc_c:
        best_acc_c = val_acc_c
        torch.save(model_c.state_dict(), os.path.join(CIFAR_OUTPUT, "best_model.pth"))

print(f"\nBest val acc: {best_acc_c:.2f}%  |  Final prune ratio: {pruner_c.get_prune_ratio():.4f}")


### 2-E  Results

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

ax1.plot(history_c["epoch"], history_c["val_acc"],    color="steelblue",  label="Val Acc (%)")
ax2.plot(history_c["epoch"], history_c["prune_ratio"], color="darkorange", linestyle="--", label="Prune Ratio")

if l0_stop_epoch is not None:
    ax1.axvline(l0_stop_epoch, color="red", linestyle=":", alpha=0.7, label=f"Early-stop (ep {l0_stop_epoch})")

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Val Accuracy (%)",  color="steelblue")
ax2.set_ylabel("Prune Ratio",        color="darkorange")
ax2.set_ylim(0, 1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
plt.title("CIFAR-10 ResNet18 2× — L0 Pruning + Fine-Tuning")
plt.tight_layout()
plt.savefig(os.path.join(CIFAR_OUTPUT, "training_curve.png"), dpi=150)
plt.show()

---
## Summary

| Dataset | Model | Target Prune | Achieved Prune | Best Val Acc |
|---------|-------|:---:|:---:|:---:|
| MNIST   | MLP 784→512→256→128→10 | ≥0.40 | (run to see) | (run to see) |
| CIFAR-10 | ResNet18 2× | ≥0.50 | **0.5049** | **95.28%** |

**Key takeaways:**
- Use `--target-prune-ratio` (early-stop) instead of running L0 training to completion.
- Stopping at exactly the target sparsity preserves well-trained weights in surviving channels.
- `merge_mask()` binarizes gates into weights; subsequent fine-tuning recovers accuracy quickly.
- `lamba=0.002` with `lamda=N` gives a good sparsity speed for both datasets.

**Equivalent CLI commands:**

```bash
# MNIST
python train.py --dataset mnist --model auto \
  --epochs 50 --batch-size 256 --lr 0.01 --optimizer adam \
  --lamba 0.002 --gate-lr-scale 1.0 --droprate-init 0.5 \
  --finetune-epochs 30 --target-prune-ratio 0.40 \
  --output-dir ./output/mnist_run

# CIFAR-10
python train.py --dataset cifar10 --model auto \
  --epochs 200 --batch-size 128 --lr 0.1 --optimizer sgd \
  --weight-decay 5e-4 --lamba 0.002 --gate-lr-scale 1.0 \
  --droprate-init 0.5 --finetune-epochs 100 \
  --target-prune-ratio 0.50 \
  --output-dir ./output/cifar10_run
```